# PDF 教材 RAG：整理版

流程：读取 PDF → 清洗与分块 → 向量化 → 写入 Chroma → 检索 → 重排序 → 交给本地大模型回答。

运行前请把 `PDF_PATH` 改成你的 PDF 文件名，并安装：`pymupdf chromadb sentence-transformers requests`。

In [ ]:
from pathlib import Path

import chromadb
import pymupdf
from sentence_transformers import CrossEncoder, SentenceTransformer

PDF_PATH = Path("test.pdf")  # 改为你的 PDF 文件路径
CHROMA_PATH = "chroma_db"
COLLECTION_NAME = "pdf_computer_network"
EMBEDDING_MODEL = "BAAI/bge-small-zh-v1.5"
RERANKER_MODEL = "BAAI/bge-reranker-base"
CHUNK_SIZE = 500
CHUNK_OVERLAP = 80

## 1. PDF 文本提取与清洗

每页保留来源文件名和页码，供后续检索结果引用。段落清洗会压缩多余空白，但不改变文本内容。

In [ ]:
def extract_pdf_pages(pdf_path: Path) -> list[dict]:
    """读取 PDF 的每一页文本，并保留来源元数据。"""
    pages = []
    with pymupdf.open(pdf_path) as document:
        for page_index, page in enumerate(document):
            text = page.get_text("text", sort=True).strip()
            if text:
                pages.append({
                    "source": pdf_path.name,
                    "page": page_index + 1,
                    "text": text,
                })
    return pages


def clean_paragraphs(text: str) -> list[str]:
    """按空行分段，删除空段并合并每段内部的换行/空格。"""
    return [
        " ".join(paragraph.split())
        for paragraph in text.split("\n\n")
        if " ".join(paragraph.split())
    ]


pages = extract_pdf_pages(PDF_PATH)
print(f"已提取 {len(pages)} 页非空文本")

## 2. 分块（Chunking）

优先按段落合并，尽量保证语义完整；超长段落再按字符切分，并保留重叠内容，减少边界处的信息丢失。

In [ ]:
def split_by_chars(text: str, chunk_size: int, overlap: int) -> list[str]:
    """将超长文本切为定长、带重叠的片段。"""
    if not 0 <= overlap < chunk_size:
        raise ValueError("overlap 必须大于等于 0 且小于 chunk_size")

    chunks, start = [], 0
    while start < len(text):
        chunk = text[start : start + chunk_size].strip()
        if chunk:
            chunks.append(chunk)
        if start + chunk_size >= len(text):
            break
        start += chunk_size - overlap
    return chunks


def chunk_paragraphs(paragraphs: list[str], max_chars: int) -> list[str]:
    """将相邻短段落合并；长段落交给字符切分。"""
    chunks, current = [], ""
    for paragraph in paragraphs:
        if len(paragraph) > max_chars:
            if current:
                chunks.append(current)
                current = ""
            chunks.extend(split_by_chars(paragraph, max_chars, CHUNK_OVERLAP))
        elif not current:
            current = paragraph
        elif len(current) + 2 + len(paragraph) <= max_chars:
            current += "\n\n" + paragraph
        else:
            chunks.append(current)
            current = paragraph
    if current:
        chunks.append(current)
    return chunks


def build_chunk_records(pages: list[dict]) -> list[dict]:
    records = []
    for page_info in pages:
        paragraphs = clean_paragraphs(page_info["text"])
        for chunk_index, text in enumerate(chunk_paragraphs(paragraphs, CHUNK_SIZE), start=1):
            records.append({
                "id": f"{PDF_PATH.stem}-p{page_info['page']}-c{chunk_index}",
                "source": page_info["source"],
                "page": page_info["page"],
                "chunk_index": chunk_index,
                "text": text,
            })
    return records


chunk_records = build_chunk_records(pages)
print(f"共生成 {len(chunk_records)} 个文本块")

## 3. 向量化并写入 Chroma

向量模型将每个文本块编码为语义向量；Chroma 持久化保存向量、原文和页码等元数据。重复运行时，`upsert` 会按相同 ID 更新记录。

In [ ]:
embedding_model = SentenceTransformer(EMBEDDING_MODEL)

ids = [record["id"] for record in chunk_records]
texts = [record["text"] for record in chunk_records]
metadatas = [
    {
        "source": record["source"],
        "page": record["page"],
        "chunk_index": record["chunk_index"],
    }
    for record in chunk_records
]
embeddings = embedding_model.encode(texts, show_progress_bar=True).tolist()

client = chromadb.PersistentClient(path=CHROMA_PATH)
collection = client.get_or_create_collection(name=COLLECTION_NAME)
collection.upsert(
    ids=ids,
    documents=texts,
    embeddings=embeddings,
    metadatas=metadatas,
)
print(f"Chroma 中现有 {collection.count()} 条记录")

## 4. 检索与重排序

先通过向量相似度取回较多候选块，再用 CrossEncoder 对“问题—文本块”逐对打分，提高最终上下文的相关性。

In [ ]:
def retrieve_and_rerank(question: str, candidate_count: int = 10, top_k: int = 3) -> list[dict]:
    query_embedding = embedding_model.encode(question).tolist()
    result = collection.query(
        query_embeddings=[query_embedding],
        n_results=candidate_count,
    )

    documents = result["documents"][0]
    metadatas = result["metadatas"][0]
    reranker = CrossEncoder(RERANKER_MODEL)
    scores = reranker.predict([[question, document] for document in documents])

    records = [
        {**metadata, "text": document, "score": float(score)}
        for metadata, document, score in zip(metadatas, documents, scores)
    ]
    return sorted(records, key=lambda record: record["score"], reverse=True)[:top_k]


question = "请说明分散式与集中式路由选择算法的区别。"
retrieved_records = retrieve_and_rerank(question)
for record in retrieved_records:
    print(f"第 {record['page']} 页，第 {record['chunk_index']} 块，得分 {record['score']:.3f}")
    print(record["text"][:200], "\n")

## 5. 组装受证据约束的提示词（可选）

这里仅生成提示词。将它传给 Ollama 或其他大模型时，要求模型只依据检索到的文本作答，并在每个要点后标注页码和分块编号。

In [ ]:
def build_rag_prompt(question: str, records: list[dict]) -> str:
    context = "\n\n".join(
        f"[来源：{item['source']}，第 {item['page']} 页，第 {item['chunk_index']} 块]\n{item['text']}"
        for item in records
    )
    return f"""你是严谨的教材问答助手。
只能使用下方参考资料中明确出现的信息回答。
每个要点后必须标注对应的来源、页码和分块；资料不足时直接说明资料不足。

问题：{question}

参考资料：
{context}
"""

prompt = build_rag_prompt(question, retrieved_records)
print(prompt)